# Turtle data preprocessing

### Imports

In [1]:

dico_info={}

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from scipy import signal
import math


## Common functions and dictionaries

In [ ]:
OUT_DIR="data_turtle/out_A_03082026"

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
else:
    print("output directory already exists")

### Possible behaviours


In [ ]:
# NOTE: these behavior names are the exact labels used by the annotation
# software and found in the "Behavior" column of the Behaviors_*.csv files.
# A few of them are in French (e.g. "Evitement", "Courant", "Nouvelle video")
# because that is how they were originally annotated. Do NOT translate them:
# the code below matches these strings exactly against the raw label files.
beh_str="""Acceleration
Basking
Breathing
Catching
Catching jellyfish
Chewing on movement
Chewing stationary
Chewing jellyfish
Escape
Evasion
Evitement
Feeding
Feeding stationary
Feeding on movement
Flipper beat
Foraging
Gliding ascent
Gliding descent
Grabbing on movement
Grabbing stationary
Grabbing the wall
Grabbing wall
Hunting jellyfish
Interaction
Landing
Nouvelle video
Obstacle
Prospection
Pursuit
Resting
Resting active
Resting in flow
Resting watching
Sand
Scratching
Scratching head
Scratching camera
Shaking
Shaking head
Head shaking
Stay in surface
Stepping back
Swimming 1 ascent
Swimming 1 descent
Swimming 1 horizontally
Swimming ascent
Swimming descent
Swimming horizontally
Swimming fastly ascent
Swimming fastly descent
Swimming fastly horizontally
Swimming on place
Swimming on the bottom
Time mark
Left U-turn
Right U-turn
Watching
Walking on sand
Courant
Regurgitating"""

In [23]:
behName_to_behInd={}
behInd_to_behName={}

for i,name in enumerate(beh_str.split("\n")):
    print(i,name)
    behName_to_behInd[name]=i
    behInd_to_behName[i]=name

behName_to_behInd["no behavior"]=-1
behInd_to_behName[-1]="no behavior"


import json
with open(OUT_DIR+"/behName_to_behInd.json", 'w') as f:
    json.dump(json.dumps(behName_to_behInd), f)

with open(OUT_DIR+"/behInd_to_behName.json", 'w') as f:
    json.dump(json.dumps(behInd_to_behName), f)



del beh_str

0 Acceleration
1 Basking
2 Breathing
3 Catching
4 Catching jellyfish
5 Chewing on movement
6 Chewing stationary
7 Chewing jellyfish
8 Escape
9 Evasion
10 Evitement
11 Feeding
12 Feeding stationary
13 Feeding on movement
14 Flipper beat
15 Foraging
16 Gliding ascent
17 Gliding descent
18 Grabbing on movement
19 Grabbing stationary
20 Grabbing the wall
21 Grabbing wall
22 Hunting jellyfish
23 Interaction
24 Landing
25 Nouvelle video
26 Obstacle
27 Prospection
28 Pursuit
29 Resting
30 Resting active
31 Resting in flow
32 Resting watching
33 Sand
34 Scratching
35 Scratching head
36 Scratching camera
37 Shaking
38 Shaking head
39 Head shaking
40 Stay in surface
41 Stepping back
42 Swimming 1 ascent
43 Swimming 1 descent
44 Swimming 1 horizontally
45 Swimming ascent
46 Swimming descent
47 Swimming horizontally
48 Swimming fastly ascent
49 Swimming fastly descent
50 Swimming fastly horizontally
51 Swimming on place
52 Swimming on the bottom
53 Time mark
54 Left U-turn
55 Right U-turn
56 Watch

### Behavior file processing functions

In [ ]:

NONE_date=pd.to_datetime("1900-01-01")


"""Warning: potential bug if the interval crosses midnight,
since the date is read from the file name (global variable DATE)."""
def concat_hour_decisecond(hour,deci):
    deci_str=str(deci)

    if hour[-2]=='60':
        res_str= DATE + ' ' + hour
    else:
        res_str= DATE + ' ' + hour+deci_str[1:5]
    try:
        res=pd.to_datetime(res_str, format='%d-%m-%Y %H:%M:%S.%f')
    except Exception as e:
        print("PROBLEM:")
        print(e)
        print(res_str)
        print(deci)
        res=NONE_date

    return res


# Note the capital 'F' in 'DeciFin', which is not present in 'DeciDeb'
# (these are the exact column names as they appear in the source Behavior CSV files)
def concat_start(line):
    return concat_hour_decisecond(line['HeureDeb'],line['DeciDeb'])


def concat_end(line):
    return concat_hour_decisecond(line['HeureFin'],line['DeciFin'])


### Resampling the labels

In [25]:
def middle_val(x):
    if len(x)==0: return -1
    else: return x[len(x)//2]

In [26]:
def resample_cat():

    serie=mes["beh"]

    serie_re=serie.resample(RESAMPLING_STEP)
    serie_re=serie_re.apply(middle_val)
    return serie_re

In [ ]:
def plot_acc_beh(df,suffix="",addVerticalBars=True):

    # default color list
    colors=plt.rcParams['axes.prop_cycle'].by_key()['color']


    fig,axs=plt.subplots(7,1,figsize=(18,9),sharex=True)

    for i,ind in enumerate(main_beh_ind):
        axs[0].plot(df.index, (df["beh"]==ind)*i,c=colors[i%len(colors)])


    yticks=[]
    for ind in main_beh_ind:
        yticks.append(str(ind)+":"+behInd_to_behName[ind])

    axs[0].set_yticks(range(len(main_beh_ind)))
    axs[0].set_yticklabels(yticks)

    axs[1].plot(df.index,df["Accelerometer X [m/s²]"],label="AccX")
    axs[1].plot(df.index,df["Accelerometer Y [m/s²]"],label="AccY")
    axs[1].plot(df.index,df["Accelerometer Z [m/s²]"],label="AccZ")
    axs[1].legend()

    axs[2].plot(df.index,df["Gyroscope X [mrad/s]"],label="GyrX")
    axs[2].plot(df.index,df["Gyroscope Y [mrad/s]"],label="GyrY")
    axs[2].plot(df.index,df["Gyroscope Z [mrad/s]"],label="GyrZ")
    axs[2].legend()


    axs[3].plot(df.index,-df["Depth [m]"],label="depth")
    axs[3].legend()

    axs[4].plot(df.index,df["StX"],label="StX")
    axs[4].plot(df.index,df["StY"],label="StY")
    axs[4].plot(df.index,df["StZ"],label="StZ")
    axs[4].legend()

    axs[5].plot(df.index,df["DX"],label="DX")
    axs[5].plot(df.index,df["DY"],label="DY")
    axs[5].plot(df.index,df["DZ"],label="DZ")
    axs[5].legend()

    axs[6].plot(df.index,df["DBA"],label="DBA")
    axs[6].legend()


    """ vertical lines indicating the interval that will be exported at the end"""
    if addVerticalBars:
        for j in [0,1]:
            if KEPT_TIME_INTERVAL[j] is not None:
                time_for_vline=pd.to_datetime(KEPT_TIME_INTERVAL[j])
                axs[0].axvline(x=time_for_vline,color="k")
                axs[1].axvline(x=time_for_vline,color="k")
                axs[2].axvline(x=time_for_vline,color="k")
                axs[3].axvline(x=time_for_vline,color="k")



    fig.tight_layout()

    fig.savefig(OUT_DIR+"/"+DIR_NAME+suffix)

In [28]:
from scipy.signal import butter, lfilter

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a


def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, data)
    return y


# Main processing loop

### Global parameters

In [ ]:
NAMES=['CC-07-107_11-02-2019',
       'CC-07-108_11-02-2019',
       'CC-07-115_13-05-2019',
       'CC-07-115_14-05-2019',
       'CC-07-47_14-02-2018',
       'CC-07-48_04-10-2018',
       'CC-07-48_06-10-2018',
       'CC-07-48_08-04-2019_1',
       'CC-07-48_08-04-2019_2',
       'CC-07-48_08-04-2019_3',
       'CC-07-48_08-10-2018',
       'CC-07-48_10-10-2018',
       'CC-07-48_18-02-2019',
       'CC-07-48_22-05-2018',
       'CC-07-48_26-05-2018',
       'CC-07-59_20-05-2018']


#NAMES=['CC-07-48_26-05-2018',
       #'CC-07-59_20-05-2018'
       #]

#NAMES=['CC-07-47_14-02-2018','CC-07-48_10-10-2018']

""" Used to remove the measurements at the end that have no associated label.
Set to None to analyze the whole measurement series."""
CUT_INDEX=400000 # or None
""" Used to remove measurements at the beginning"""
CUT_INDEX_BEGIN=None

""" Used to shift the measurement series in time.
A value is computed automatically further below, but you should check that this
computation actually gives a sensible result."""

BEH_COL_NAME="Behavior"

""" Used to remove an atypical beginning or end.
Warning: this must be expressed in the final time zone (the one seen on the final plots)."""
KEPT_TIME_INTERVAL=(None,None)

delim=","
Pressure=None

RESAMPLING_STEP="50 ms" # "100 ms"  #250 ms #62.5 ms
RESAMPLING_STEP_S=dico_info["RESAMPLING_STEP_S"]=float(RESAMPLING_STEP[:-3])/1000


### Main loop

In [ ]:

print("resampling interval in second",RESAMPLING_STEP_S)

for name in NAMES:
    DIR_NAME=name
    delim=","
    Pressure=None
    CUT_INDEX_BEGIN=None
    KEPT_TIME_INTERVAL=(None,None)

    # Retrieve global parameters for this turtle/session
    #1/
    if DIR_NAME=='CC-07-107_11-02-2019':
        SAMPLING_STEP=50
        CUT_INDEX=354802
        MES_TIME_DECAY=None
        KEPT_TIME_INTERVAL=("2019-02-11 11:54:37",None)
        Ax=-0.09396204
        Ay=-0.02074487
        Az=-1.047006
        Base_d=0.4
        FREQ=20

        DATE="11-02-2019"

    #2/
    elif DIR_NAME=='CC-07-108_11-02-2019':
        SAMPLING_STEP=50
        CUT_INDEX=160952
        MES_TIME_DECAY=None
        KEPT_TIME_INTERVAL=("2019-02-11 12:09:15",None)
        Ax=0.04929957
        Ay=-0.006345489
        Az=-1.062625
        Base_d=0.59
        FREQ=20
        DATE="11-02-2019"

    #3/
    elif DIR_NAME=='CC-07-115_13-05-2019':
        SAMPLING_STEP=50 
        CUT_INDEX=734326
        MES_TIME_DECAY=None
        KEPT_TIME_INTERVAL=('2019-05-13 08:31:29',None)
        Ax=-0.01220286
        Ay=0.1100698
        Az=-1.071655
        Base_d=0.5
        FREQ=20
        DATE="13-05-2019"

    #4/
    elif DIR_NAME=='CC-07-115_14-05-2019':
        SAMPLING_STEP=50 
        CUT_INDEX=778609
        MES_TIME_DECAY=None
        KEPT_TIME_INTERVAL=('2019-05-14 06:18:00',None)
        Ax=-0.01220286
        Ay=0.1100698
        Az=-1.071655
        Base_d=0.5
        FREQ=20
        DATE="14-05-2019"

    #5/
    elif DIR_NAME=='CC-07-47_14-02-2018':
        SAMPLING_STEP=20 
        CUT_INDEX=294855
        MES_TIME_DECAY=-1
        KEPT_TIME_INTERVAL=('2018-02-14 14:16:00',None)
        Ax=0.1403329
        Ay=0.1037243
        Az=-0.958901
        Base_d=10.17
        FREQ=50
        DATE="14-02-2018"

    #6/
    elif DIR_NAME=='CC-07-48_04-10-2018':
        SAMPLING_STEP=50
        CUT_INDEX=322321
        MES_TIME_DECAY=None
        KEPT_TIME_INTERVAL=('2018-10-04 09:46:07',None)
        Ax=-0.26
        Ay=-0.02
        Az=-0.83
        Base_d=10.3
        FREQ=20
        DATE="04-10-2018"



    #7/
    elif DIR_NAME=='CC-07-48_06-10-2018':
        SAMPLING_STEP=50
        CUT_INDEX=483303
        MES_TIME_DECAY=-1
        KEPT_TIME_INTERVAL=('2018-10-06 11:23:54',None)
        Ax=-0.36
        Ay=0.02
        Az=-0.75
        Base_d=10.3
        FREQ=20
        DATE="06-10-2018"


    #8/
    elif DIR_NAME=='CC-07-48_08-04-2019_1':
        SAMPLING_STEP=50
        CUT_INDEX=165535
        MES_TIME_DECAY=None
        KEPT_TIME_INTERVAL=("2019-04-08 08:54:00",None)
        Ax=-0.2042759
        Ay=0.05759751
        Az=-0.8241813
        Base_d=0
        FREQ=20
        DATE="08-04-2019"
        delim=";"

    #9/
    elif DIR_NAME=='CC-07-48_08-04-2019_2':
        SAMPLING_STEP=50
        CUT_INDEX=None
        MES_TIME_DECAY=None
        KEPT_TIME_INTERVAL=("2019-04-08 12:31:00",None)
        Ax=-0.2042759
        Ay=0.05759751
        Az=-0.8241813
        Base_d=0
        FREQ=20
        DATE="08-04-2019"
        delim=";"

    #10/
    elif DIR_NAME=='CC-07-48_08-04-2019_3':
        SAMPLING_STEP=50
        CUT_INDEX=None
        MES_TIME_DECAY=None
        KEPT_TIME_INTERVAL=("2019-04-08 14:03:00","2019-04-08 16:02:00")
        Ax=-0.2042759
        Ay=0.05759751
        Az=-0.8241813
        Base_d=0
        FREQ=20
        DATE="08-04-2019"
        delim=";"

    #11/
    elif DIR_NAME=='CC-07-48_08-10-2018':
        SAMPLING_STEP=50
        CUT_INDEX=574733
        MES_TIME_DECAY=-6
        KEPT_TIME_INTERVAL=("2018-10-08 10:34:00","2018-10-08 18:00:00")
        Ax=-0.37
        Ay=-0.05
        Az=-0.72
        Base_d=10.3
        FREQ=20
        DATE="08-10-2018"
        delim=","

    #12/
    elif DIR_NAME=='CC-07-48_10-10-2018':
        SAMPLING_STEP=50
        CUT_INDEX=105599
        MES_TIME_DECAY=-6
        KEPT_TIME_INTERVAL=("2018-10-10 09:48:36",None)
        Ax=-0.2550398
        Ay=0.008053889
        Az=-0.7538929
        Base_d=10.3
        FREQ=20
        DATE="10-10-2018"

    #13/
    elif DIR_NAME=='CC-07-48_18-02-2019':
        SAMPLING_STEP=50
        CUT_INDEX=584679
        MES_TIME_DECAY=None
        KEPT_TIME_INTERVAL=("2019-02-18 10:16:29",None)
        Ax=-0.2052522
        Ay=-0.09542639
        Az=-0.8139309
        Base_d=0
        FREQ=20
        DATE="18-02-2019"
        Pressure=-1

    #14/
    elif DIR_NAME=='CC-07-48_22-05-2018':
        SAMPLING_STEP=20
        CUT_INDEX=715401
        MES_TIME_DECAY=None
        # no need for KEPT_TIME_INTERVAL here (we keep everything)
        Ax=-0.1818227
        Ay=-0.04002539
        Az=-0.8222289
        Base_d=10.3
        FREQ=50
        DATE="22-05-2018"
        Pressure=None

    #15 STILL VERY STRANGE./
    elif DIR_NAME=='CC-07-48_26-05-2018':
        SAMPLING_STEP=20
        CUT_INDEX=903254
        CUT_INDEX_BEGIN=130000 # for the raw file
        MES_TIME_DECAY=-6
        KEPT_TIME_INTERVAL=("2018-05-26 11:02:00","2018-05-26 14:44:45")
        Ax=-0.2936009
        Ay=-0.06540734
        Az=-0.7765902
        Base_d=10.3
        FREQ=50
        DATE="26-05-2018"


    #16/
    elif DIR_NAME=='CC-07-59_20-05-2018':
        SAMPLING_STEP=20
        CUT_INDEX=1114236
        MES_TIME_DECAY=None
        KEPT_TIME_INTERVAL=("2018-05-20 08:45:23",None)
        Ax=-0.05515694
        Ay=-0.01586372
        Az=-0.9962417
        Base_d=10.3
        FREQ=50
        DATE="20-05-2018"
        delim=";"


    #17/
    elif DIR_NAME=='CC-07-59_25-05-2018':
        SAMPLING_STEP=20
        CUT_INDEX=1125761
        MES_TIME_DECAY=None
        KEPT_TIME_INTERVAL=("2018-05-25 09:13:02",None)
        Ax=-0.1117782
        Ay=-0.1232489
        Az=-1.017231
        Base_d=10.3
        FREQ=50
        DATE="25-05-2018"
    else:
        assert 1==0, "not implemented yet"


    print("DATE:",DATE)
    BEHAVIORS_FILE_PATH='Behavior/Behaviors_'+DIR_NAME+'.csv'
    print("Behavior Path",BEHAVIORS_FILE_PATH)
    ACCELERO_FILE_PATH='Data_Accelero/'+DIR_NAME+'.csv'
    print('Accelero Path', ACCELERO_FILE_PATH)

# Processing the behavior file
    df=pd.read_csv(BEHAVIORS_FILE_PATH,delimiter=";")
    df['datetime_deb']=df.apply(concat_start,axis=1)
    df['datetime_fin']=df.apply(concat_end,axis=1)
    beh_df=df[["datetime_deb","datetime_fin",BEH_COL_NAME]].copy()
    beh_df[BEH_COL_NAME].replace(behName_to_behInd,inplace=True)
    len(df)

# Processing the measurement file
## Import and initial trimming
    print("file:",ACCELERO_FILE_PATH)
    if name=='CC-07-48_26-05-2018' :
        mesures_df= pd.read_csv(ACCELERO_FILE_PATH,delimiter=delim)
    else :
        mesures_df= pd.read_csv(ACCELERO_FILE_PATH,delimiter=delim,encoding = "ISO-8859-1")

    time_data_change_name_dict = {' Time (local)':'Time (local)', " Date (local)":"Date (local)"}
    try:
        mesures_df.rename(columns=time_data_change_name_dict, inplace=True)
        print("Done")
    except Exception:
        print("Names were not changed. Probably it is already fine.")
    try:
        mesures_df=mesures_df[["Date (local)","Time (local)","Accelerometer X [m/s²]","Accelerometer Y [m/s²]","Accelerometer Z [m/s²]","Gyroscope X [mrad/s]","Gyroscope Y [mrad/s]","Gyroscope Z [mrad/s]","Magnetometer X [µT]","Magnetometer Y [µT]","Magnetometer Z [µT]","Depth (100bar) 1 [m]","Light intensity 1 [raw]","Temperature (imu) [°C]"]].copy()
        mesures_df.rename(columns={"Depth (100bar) 1 [m]":"Depth [m]"}, inplace=True)
        print("Done 1.")
    except Exception:
        print("Depth != 100")
    try:
        mesures_df=mesures_df[["Date (local)","Time (local)","Accelerometer X [m/s²]","Accelerometer Y [m/s²]","Accelerometer Z [m/s²]","Gyroscope X [mrad/s]","Gyroscope Y [mrad/s]","Gyroscope Z [mrad/s]","Magnetometer X [µT]","Magnetometer Y [µT]","Magnetometer Z [µT]","Depth (30bar) 1 [m]","Light intensity 1 [raw]","Temperature (imu) [°C]"]].copy()
        mesures_df.rename(columns={"Depth (30bar) 1 [m]":"Depth [m]"}, inplace=True)
        print("Done 2.")
    except Exception:
        print("Depth != 30")
    mesures_df.head()

## CUT INDEX
    print("the manually entered CUT_INDEX is:",CUT_INDEX)

    if CUT_INDEX is None:
        CUT_INDEX=-1
    else:
        print("measurements will be cut at the end at index:",CUT_INDEX)

    if CUT_INDEX_BEGIN is None:
        CUT_INDEX_BEGIN=0
    else:
        print("measurements will be cut at the beginning at index:",CUT_INDEX_BEGIN)

    """.copy() to avoid a lot of warnings later on"""
    mesure=mesures_df[CUT_INDEX_BEGIN:CUT_INDEX].copy()
    nb_suppr=len(mesures_df)-len(mesure)
    print("number of measurements removed:",nb_suppr)
## Time indexing
    %time
    mesure['datetime']=pd.to_datetime(mesure['Date (local)'] + ' ' + mesure['Time (local)'],format='%d.%m.%Y %H:%M:%S.%f')
    mesure.drop(['Date (local)','Time (local)'], axis=1, inplace=True)

## Pressure and accelerometer correction
### Pressure
    if Pressure is not None:
        mesure.iloc[:,9]=mesure.iloc[:,9]*Pressure
    mesure.iloc[:,9]= mesure.iloc[:,9]-Base_d
    mesure['Depth [m]']=mesure['Depth [m]'].apply(lambda x : x if x > 0 else 0)
### Accelerometer

# compute DBA (Dynamic Body Acceleration)
    mask_length=int(2*(1/SAMPLING_STEP)*1000)
    mask = np.ones(mask_length)/mask_length

    AccX_dc=mesure.iloc[:,0].copy()
    AccY_dc=mesure.iloc[:,1].copy()
    AccZ_dc=mesure.iloc[:,2].copy()
# smoothing
    #AccX=signal.sosfilt(signal.butter(4, 0.25, 'lp', output = 'sos'), AccX_dc)
    #AccY=signal.sosfilt(signal.butter(4, 0.25, 'lp', output = 'sos'), AccY_dc)
    #AccZ=signal.sosfilt(signal.butter(4, 0.25, 'lp', output = 'sos'), AccZ_dc)

    StX = np.convolve(AccX_dc, mask, mode='same')
    StY = np.convolve(AccY_dc, mask, mode='same')
    StZ = np.convolve(AccZ_dc, mask, mode='same')

    DX=AccX_dc-StX
    DY=AccY_dc-StY
    DZ=AccZ_dc-StZ
    dba = np.sqrt(DX**2 + DY**2 + DZ**2)


    AccX_high=signal.sosfilt(signal.butter(1, [0.10,0.25], 'bp', output = 'sos'), AccX_dc)
    AccY_high=signal.sosfilt(signal.butter(1, [0.10,0.25], 'bp', output = 'sos'), AccY_dc)
    AccZ_high=signal.sosfilt(signal.butter(1, [0.10,0.25], 'bp', output = 'sos'), AccZ_dc)


# Compute RA (Rotational Acceleration / gyroscope norm)

    #GyrX=mesure.iloc[:,3].copy()
    #GyrY=mesure.iloc[:,4].copy()
    #GyrZ=mesure.iloc[:,5].copy()
    #ra = np.sqrt(GyrX**2 + GyrY**2 + GyrZ**2)


    #mesure['AccX_smooth']=AccX_dc
    #mesure['AccY_smooth']=AccY_dc
    #mesure['AccZ_smooth']=AccZ_dc
    mesure['StX']=StX
    mesure['StY']=StY
    mesure['StZ']=StZ
    mesure['DX']=DX
    mesure['DY']=DY
    mesure['DZ']=DZ
    mesure['dba']=dba
    mesure['AccX_high']=AccX_high
    mesure['AccY_high']=AccY_high
    mesure['AccZ_high']=AccZ_high
    #mesure['RA']=ra
    #mesure['GyrX_high']=GyrX
    #mesure['GyrY_high']=GyrY
    #mesure['GyrZ_high']=GyrZ

    print(mesure.head())

## Compute and optionally apply the MES_TIME_DECAY (time offset)
    if MES_TIME_DECAY is not None:
        delta=pd.Timedelta(MES_TIME_DECAY, unit='h')
        print("time offset applied:",delta)
        mes=mesure.copy()
        mes.datetime=mes.datetime+delta
        del mesure
    else:
        mes=mesure
## Time becomes the dataframe index
    mes.set_index('datetime', inplace=True)


## Resampling pressure
    mes['Pressure_corr']=np.NAN
    mes.iloc[np.where(np.diff(mes.iloc[:,9])!=0)[0],-1]=mes.iloc[np.where(np.diff(mes.iloc[:,9])!=0)[0],9]
    mes['Pressure_corr'].interpolate(method='linear',inplace=True)
    mes['Pressure_corr'].fillna(method="bfill",inplace=True)


    mes['Pressure_diff']=np.NAN
    #mes.iloc[np.where(np.diff(mes.iloc[:,9])!=0)[0],-1]=np.concatenate((np.diff(mes.iloc[np.where(np.diff(mes.iloc[:,9])!=0)[0],9]),np.array([0])),axis=0)
    mes.iloc[np.where(np.diff(mes.iloc[:,9])!=0)[0],-1]=mes.iloc[np.where(np.diff(mes.iloc[:,9])!=0)[0],9].diff()
    mes['Pressure_diff'].interpolate(method='linear',inplace=True)
    mes['Pressure_diff'].fillna(method="bfill",inplace=True)


## Merge the behaviors and measurements
    """create new columns"""
    mes["beh"]=-1 #-> "no behavior"
    sum_inter=0
    max_inter=0
    index_max_inter=0
    nb_outside=0
    i=-1
    for index,row in beh_df.iterrows():
        i+=1
        deb=row["datetime_deb"]
        fin=row["datetime_fin"]
        beh=row[BEH_COL_NAME]
        inter= sum(mes.loc[deb:fin,"beh"]!=-1)
        #print(inter,i)
        if inter>max_inter:
            max_inter=inter
            index_max_inter=i
        sum_inter+=inter
        mes.loc[deb:fin,"beh"]=beh
        if len(mes.loc[deb:fin])==0:
            nb_outside+=1
            print("time interval " +str(deb)+"->"+str(fin)+" from the behavior file does not correspond to any interval in the measurements")
    print("number of labels outside of the series:"+str(nb_outside)+" out of:"+str(len(beh_df)))

# Resampling
## Resampling the labels
    labels_beh_re=resample_cat()
## Resampling the measurements
    mes_sub=mes.resample(RESAMPLING_STEP).mean()
    mes_sub["beh"]=labels_beh_re

# Plot and save figures
    main_beh_vc=mes_sub["beh"].value_counts()[:17]
    main_beh_ind=list(main_beh_vc.index)
    main_beh_str=[behInd_to_behName[val] for val in main_beh_ind]
    list(zip(main_beh_ind,main_beh_str,main_beh_vc.values))
    #plot_acc_beh(mes_sub)

# Export
    dico_info["col_names"]=list(mes_sub.columns.values)
    if KEPT_TIME_INTERVAL[0] is not None:
        print("trimming the beginning")
        begin=pd.to_datetime(KEPT_TIME_INTERVAL[0])
    else:
        begin=mes_sub.index[0]
    if KEPT_TIME_INTERVAL[1] is not None:
        print("trimming the end")
        end=pd.to_datetime(KEPT_TIME_INTERVAL[1])
    else:
        end=mes_sub.index[-1]
    print("start:",begin)
    print("end:",end)
    print("duration",end-begin)
    mes_matrix=mes_sub[begin:end].values
    np.save(OUT_DIR+"/"+DIR_NAME+'.npy',mes_matrix)

    with open(OUT_DIR+"/dico_info.json", 'w') as f:
        json.dump(json.dumps(dico_info), f)
